In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IPL Silver Layer") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/15 15:04:33 WARN Utils: Your hostname, LAPTOP-T50I9I4T, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/15 15:04:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 15:04:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/15 15:04:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/15 15:04:35 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/15 15:04:35 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [3]:
df = spark.read.parquet('../data/bronze/ipl')
df.show(10)
df.printSchema()

26/03/15 15:04:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+----------+-------------------+--------------------+-------+--------------------+--------------------+----+----+-------+-----------+-------+-----------+-----------+-------+----------+-----------+----------+-----------+-----------------+----------+-----------+---------------+-----------+----------+--------+-----------+-------------+-------------+---------------+------+------------+---------------+--------------------+-----------+--------------------+-------------+--------------------+---------+---+-----+----+-------+------+---------+----------------+-----------+------+--------------+-----+--------------+-------+------------+---------+----------+-----------+----------+-----------+------------+-------------+--------------------+-----------+-----------+
|match_id|      date|         match_type|          event_name|innings|        batting_team|        bowling_team|over|ball|ball_no|     batter|bat_pos|runs_batter|balls_faced| bowler|valid_ball|runs_extras|runs_total|runs_bowler|ru

### Basic Data cleaning and Data Transformation

In [4]:
df = df.withColumn("match_type",col("match_type").cast("string"))


In [9]:
df.printSchema()

root
 |-- match_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- match_type: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- innings: integer (nullable = true)
 |-- batting_team: string (nullable = true)
 |-- bowling_team: string (nullable = true)
 |-- over: integer (nullable = true)
 |-- ball: integer (nullable = true)
 |-- ball_no: double (nullable = true)
 |-- batter: string (nullable = true)
 |-- bat_pos: integer (nullable = true)
 |-- runs_batter: integer (nullable = true)
 |-- balls_faced: integer (nullable = true)
 |-- bowler: string (nullable = true)
 |-- valid_ball: integer (nullable = true)
 |-- runs_extras: integer (nullable = true)
 |-- runs_total: integer (nullable = true)
 |-- runs_bowler: integer (nullable = true)
 |-- runs_not_boundary: boolean (nullable = true)
 |-- extra_type: string (nullable = true)
 |-- non_striker: string (nullable = true)
 |-- non_striker_pos: integer (nullable = true)
 |-- wicket_kind: string (nullable

##### Checking if the shot is boundry or not

In [5]:
df = df.withColumn("is_boundary",when(col("runs_batter")> 4 , 1).otherwise(0))

##### Wicket Check

In [6]:
df = df.withColumn("is_wicket",when(col("wicket_kind").isNotNull(),1).otherwise(0))

##### filtering only Legal Delivery

In [7]:
df = df.filter(col("valid_ball") == 1)

##### Only Selecting required rows

In [8]:
df_silver = df.select(
    "match_id",
    "batting_team",
    "bowling_team",
    "batter",
    "bowler",
    "over",
    "ball",
    "runs_batter",
    "runs_total",
    "is_boundary",
    "is_wicket",
    "venue",
    "year"
) 

In [9]:
df_silver.show(10)

+--------+--------------------+--------------------+-----------+-------+----+----+-----------+----------+-----------+---------+--------------------+----+
|match_id|        batting_team|        bowling_team|     batter| bowler|over|ball|runs_batter|runs_total|is_boundary|is_wicket|               venue|year|
+--------+--------------------+--------------------+-----------+-------+----+----+-----------+----------+-----------+---------+--------------------+----+
|  335982|Kolkata Knight Ri...|Royal Challengers...| SC Ganguly|P Kumar|   0|   1|          0|         1|          0|        0|M Chinnaswamy Sta...|2008|
|  335982|Kolkata Knight Ri...|Royal Challengers...|BB McCullum|P Kumar|   0|   2|          0|         0|          0|        0|M Chinnaswamy Sta...|2008|
|  335982|Kolkata Knight Ri...|Royal Challengers...|BB McCullum|P Kumar|   0|   3|          0|         0|          0|        0|M Chinnaswamy Sta...|2008|
|  335982|Kolkata Knight Ri...|Royal Challengers...|BB McCullum|P Kumar|   0

###  Writing to Silver 

In [10]:
df_silver.write.mode("overwrite")\
                    .parquet("../data/silver/ipl_cleaned")